# Model Card -> Agent Engineering Workbook
## Qwen3-8B as the worked example, any model as the transferable skill

### Objective

This workbook is not primarily about Qwen3-8B. Qwen3-8B is the specimen.

The real skill you are training is:

> **Read a model card -> extract the model contract -> infer capabilities and constraints -> choose the smallest effective agent loop -> design interfaces -> define tests.**

By the end, you should be able to take an unfamiliar model card and produce an agent architecture without blindly copying a framework tutorial.

### Operating principle

**MODEL PROPERTY -> AGENT REQUIREMENT -> ARCHITECTURAL DECISION -> IMPLEMENTATION -> EVALUATION**

We will use Qwen3-8B for the worked examples, then generalize the same method to an imaginary model called `NewModel-X`.


## 0. Ground truth: read the official card first

For this workbook, use the official sources:

- Hugging Face: https://huggingface.co/Qwen/Qwen3-8B
- Qwen concepts: https://github.com/QwenLM/Qwen3/blob/main/docs/source/getting_started/concepts.md
- Qwen function calling: https://github.com/QwenLM/Qwen3/blob/main/docs/source/framework/function_call.md
- Qwen3 technical report: https://arxiv.org/abs/2505.09388

A strong engineer reads the **model card + model-specific usage docs + serving/tool docs** as one model contract.


# 1. Exercise: build the model bio

Do not write an agent yet.

First compress the model card into a small, architecture-relevant bio. Avoid collecting trivia.

### Qwen3-8B: fill the bio

| Dimension | Your notes |
|---|---|
| Exact checkpoint | |
| Family | |
| Base / instruct / chat | |
| Architecture | |
| Parameters / active parameters | |
| Context | |
| Modalities | |
| Languages | |
| Reasoning behavior | |
| Tool calling | |
| Structured output | |
| Deployment options | |
| License | |
| Key limitations / caveats | |

### Rule

If a model-card fact cannot plausibly change an agent design, latency/cost decision, interface, or evaluation plan, it is probably lower priority during first-pass reconnaissance.


In [1]:
qwen3_bio = {
    "checkpoint": "Qwen/Qwen3-8B",
    "family": "Qwen3",
    "architecture": "dense causal language model",
    "parameters": 8.2e9,
    "non_embedding_parameters": 6.95e9,
    "layers": 36,
    "attention": "GQA: 32 Q heads / 8 KV heads",
    "native_context": 32768,
    "validated_long_context": 131072,
    "languages": "100+ languages/dialects",
    "reasoning": "thinking + non-thinking modes",
    "tool_use": "agentic/tool calling support",
    "serving": ["Transformers", "vLLM", "SGLang", "Ollama", "LM Studio", "MLX-LM", "llama.cpp"],
    "license": "Apache-2.0",
}
qwen3_bio


{'checkpoint': 'Qwen/Qwen3-8B',
 'family': 'Qwen3',
 'architecture': 'dense causal language model',
 'parameters': 8200000000.0,
 'non_embedding_parameters': 6950000000.0,
 'layers': 36,
 'attention': 'GQA: 32 Q heads / 8 KV heads',
 'native_context': 32768,
 'validated_long_context': 131072,
 'languages': '100+ languages/dialects',
 'reasoning': 'thinking + non-thinking modes',
 'tool_use': 'agentic/tool calling support',
 'serving': ['Transformers',
  'vLLM',
  'SGLang',
  'Ollama',
  'LM Studio',
  'MLX-LM',
  'llama.cpp'],
 'license': 'Apache-2.0'}





## 2. Exercise: separate capability from system property

Complete the statements.

**Model capability:** __________________________

**Agent-system property:** _____________________

Use Qwen3's reasoning as the example.

A useful distinction:

- "The model can generate a thinking trace" is a model capability.
- "The agent reasons reliably on my workload" is a system property.

Reliable reasoning may require planning, tools, state, verification, retries, and deterministic code around the model.


In [2]:
capability_hypotheses = [
    {
        "claim": "Thinking mode is available",
        "capability": "The model can spend more inference on complex reasoning",
        "system_property_to_prove": "Thinking improves task success enough to justify its latency/token cost",
        "test": "Compare matched tasks in thinking vs non-thinking mode",
    },
    {
        "claim": "Tool calling is supported",
        "capability": "The model can request external actions in the expected format",
        "system_property_to_prove": "The complete model-parser-executor loop executes the right tool with correct arguments",
        "test": "Measure tool selection and argument accuracy separately",
    },
]
for h in capability_hypotheses:
    print(h)


{'claim': 'Thinking mode is available', 'capability': 'The model can spend more inference on complex reasoning', 'system_property_to_prove': 'Thinking improves task success enough to justify its latency/token cost', 'test': 'Compare matched tasks in thinking vs non-thinking mode'}
{'claim': 'Tool calling is supported', 'capability': 'The model can request external actions in the expected format', 'system_property_to_prove': 'The complete model-parser-executor loop executes the right tool with correct arguments', 'test': 'Measure tool selection and argument accuracy separately'}


# 3. Exercise: discover the model's "hidden contract"

A model card often leaves critical details outside the headline specs.

Search the documentation for:

1. Chat template
2. Role/message structure
3. Reasoning switches
4. Tool schema
5. Tool-result representation
6. Stop/special tokens
7. Structured output format
8. Serving-time parser requirements
9. Recommended sampling
10. Context extension / long-context instructions

### Qwen3 findings to record

- `enable_thinking=True/False`
- `/think` and `/no_think`
- `apply_chat_template(...)`
- reasoning parser requirements in serving examples
- Qwen-Agent as a reference integration
- tool protocol / function-call structure
- special handling around thinking content and conversation history

This is where a "good model" can become a "bad agent" because of interface mistakes.


In [3]:
model_adapter_contract = {
    "message_formatter": "model-specific chat template",
    "reasoning_control": "model-specific switch or prompt convention",
    "tool_schema": "model-specific tool representation",
    "tool_result_parser": "model-specific parser",
    "output_parser": "final-answer / structured-output parser",
    "serving_options": "model + framework specific",
}
model_adapter_contract


{'message_formatter': 'model-specific chat template',
 'reasoning_control': 'model-specific switch or prompt convention',
 'tool_schema': 'model-specific tool representation',
 'tool_result_parser': 'model-specific parser',
 'output_parser': 'final-answer / structured-output parser',
 'serving_options': 'model + framework specific'}

# 4. Exercise: reasoning-mode routing

Qwen3 exposes a powerful architectural option: one checkpoint can switch between thinking and non-thinking behaviors.

Your job is **not** to enable thinking everywhere.

Your job is to define a policy.

### Route to FAST / NON-THINKING when:
- the task is simple
- the answer is mostly transformation or retrieval presentation
- latency dominates
- additional deliberation is unlikely to change the result

### Route to THINKING when:
- the task is multi-step
- tool choice requires planning
- the answer needs verification
- coding/math/logical reasoning is central
- failure is costly enough to justify more inference

### Key question

What signal will your router use?

Possible signals:
- explicit task class
- estimated difficulty
- presence of tools
- number of required steps
- user-requested depth
- historical failure rate
- budget / latency constraints

Do not hard-code "long prompt = hard problem." That proxy is often weak.


In [4]:
def choose_reasoning_mode(task, complexity, has_tools=False, high_risk=False):
    if high_risk:
        return "thinking"
    if has_tools and complexity >= 2:
        return "thinking"
    if complexity >= 3:
        return "thinking"
    return "non-thinking"

examples = [
    {"task": "Rewrite text", "complexity": 1, "has_tools": False, "high_risk": False},
    {"task": "Compare 4 products using web data", "complexity": 3, "has_tools": True, "high_risk": False},
    {"task": "Multi-step research synthesis", "complexity": 4, "has_tools": True, "high_risk": True},
]
for ex in examples:
    mode = choose_reasoning_mode(ex["task"], ex["complexity"], ex["has_tools"], ex["high_risk"])
    print(f"{ex['task']}: {mode}")


Rewrite text: non-thinking
Compare 4 products using web data: thinking
Multi-step research synthesis: thinking


## 4A. Design the mode policy as a cost/quality tradeoff

Do not ask only "which mode is smarter?"

Ask:

> **What is the marginal value of more reasoning?**

A useful experiment matrix:

| Task class | Non-thinking success | Thinking success | Non-thinking latency | Thinking latency | Decision |
|---|---:|---:|---:|---:|---|
| Simple QA | | | | | |
| Classification | | | | | |
| Tool selection | | | | | |
| Multi-hop research | | | | | |
| Coding | | | | | |

The goal is not maximum thinking. The goal is **optimal reasoning allocation**.


# 5. Exercise: design the smallest useful agent loop

Start with the simplest thing that can work.

### Level 0 - direct model

```text
USER -> MODEL -> ANSWER
```

### Level 1 - single tool

```text
USER -> MODEL -> TOOL -> RESULT -> MODEL -> ANSWER
```

### Level 2 - iterative agent

```text
USER -> MODEL -> TOOL -> RESULT -> MODEL -> TOOL -> RESULT -> MODEL -> ANSWER
```

### Level 3 - verified agent

```text
USER -> PLAN -> ACT -> OBSERVE -> CHECK -> (ACT or ANSWER)
```

Do not begin with a multi-agent swarm.

A strong single agent with good tools, context management, retries, and verification is often the best first architecture.


In [5]:
from dataclasses import dataclass, field
from typing import Any

@dataclass
class AgentState:
    user_request: str
    messages: list[dict] = field(default_factory=list)
    plan: str | None = None
    observations: list[Any] = field(default_factory=list)
    tool_calls: int = 0
    steps: int = 0
    final_answer: str | None = None

state = AgentState("Find why company X's revenue fell.")
state


AgentState(user_request="Find why company X's revenue fell.", messages=[], plan=None, observations=[], tool_calls=0, steps=0, final_answer=None)

# 6. Exercise: tool use is a control-loop problem

Never stop at:

> "The model supports tools."

Break tool use into independent capabilities:

1. Tool discovery / availability awareness
2. Tool selection
3. Argument construction
4. Execution
5. Result ingestion
6. Follow-up decision
7. Error recovery
8. Termination

A model can be excellent at #2 and poor at #3.
It can be excellent at #3 and poor at #8.

So test each part.

### Qwen3-specific architectural lesson

The official Qwen documentation describes multi-step and parallel tool use, and Qwen recommends a model-specific integration path such as Qwen-Agent.

Therefore, do not assume a generic ReAct prompt or generic parser is automatically the best interface.


In [6]:
def execute_tool(name, args):
    tools = {
        "search": lambda q: {"results": [f"Result for: {q}"]},
        "calculator": lambda expression: {"value": eval(expression, {"__builtins__": {}}, {})},
    }
    if name not in tools:
        return {"error": f"unknown tool: {name}"}
    try:
        return tools[name](**args)
    except Exception as e:
        return {"error": str(e)}

trajectory = [
    {"kind": "model", "content": "I need evidence about the revenue decline."},
    {"kind": "tool_call", "name": "search", "args": {"q": "Company X revenue decline causes"}},
    {"kind": "tool_result", "result": execute_tool("search", {"q": "Company X revenue decline causes"})},
    {"kind": "model", "content": "I have evidence. I will synthesize and answer."},
]
trajectory


[{'kind': 'model', 'content': 'I need evidence about the revenue decline.'},
 {'kind': 'tool_call',
  'name': 'search',
  'args': {'q': 'Company X revenue decline causes'}},
 {'kind': 'tool_result',
  'result': {'results': ['Result for: Company X revenue decline causes']}},
 {'kind': 'model',
  'content': 'I have evidence. I will synthesize and answer.'}]

# 7. Exercise: parallel tool calling

Parallelism is not just a performance trick.

### Independent tools

```text
          -> Search A -
MODEL ----> Search B ----> MODEL
          -> Search C -
```

### Dependent tools

```text
MODEL -> Search A -> derive query -> Search B -> MODEL
```

The first can often be parallelized. The second generally cannot.

Therefore, when a model card says "parallel tool calls," ask:

> **Under what dependency conditions can my executor safely parallelize them?**

The model proposes actions. Your runtime decides whether those actions are safe and independent.


In [7]:
def can_parallelize(calls):
    return all(not call.get("depends_on") for call in calls)

calls = [
    {"tool": "search", "args": {"q": "A"}, "depends_on": None},
    {"tool": "search", "args": {"q": "B"}, "depends_on": None},
    {"tool": "search", "args": {"q": "C"}, "depends_on": None},
]
print("Safe to parallelize:", can_parallelize(calls))


Safe to parallelize: True


# 8. Exercise: context engineering

A context window is a **budget**, not a free memory pool.

Build a budget:

```text
TOTAL CONTEXT
- system instructions
- tool definitions
- current request
- conversation state
- retrieved evidence
- tool outputs
- reasoning / working state
- reserved generation budget
= usable slack
```

For Qwen3-8B, the official card states a native 32,768-token context and validates up to 131,072 tokens with YaRN.

The architecture question is not:

> "Can the model fit 131K?"

It is:

> "What should be in context at each step, and what should live outside context?"


In [8]:
def context_budget(total, system=0, tools=0, history=0, retrieval=0, tool_results=0, working=0, reserve_output=0):
    used = system + tools + history + retrieval + tool_results + working + reserve_output
    return {
        "total": total,
        "used": used,
        "remaining": total - used,
        "utilization_pct": round(100 * used / total, 2) if total else None,
    }

budget = context_budget(
    total=32768,
    system=2500,
    tools=5000,
    history=6000,
    retrieval=9000,
    tool_results=4500,
    working=2500,
    reserve_output=32768 // 5,
)
budget


{'total': 32768, 'used': 36053, 'remaining': -3285, 'utilization_pct': 110.03}

## 8A. Design a memory boundary

Separate:

### Working state
Temporary information needed to complete the current task:
- active plan
- recent observations
- current tool results
- temporary hypotheses

### Durable memory
Information worth carrying across tasks:
- user preferences
- stable facts
- validated long-term state
- prior outcomes that should influence future decisions

### Retrieval store
Information that is too large or dynamic to keep in the prompt:
- documents
- knowledge bases
- web pages
- logs
- databases

This separation prevents "just put everything in the context window" architecture.


# 9. Exercise: convert the model card into an architecture matrix

Fill this for Qwen3, then reuse the same table for every new model.

| Model-card statement | Evidence | Capability | Risk / constraint | Design consequence | Test |
|---|---|---|---|---|---|
| Thinking mode | | | | | |
| Non-thinking mode | | | | | |
| Tool calling | | | | | |
| Parallel tools | | | | | |
| Context | | | | | |
| Chat template | | | | | |
| Serving parser | | | | | |
| Long-context extension | | | | | |
| Recommended decoding | | | | | |

The discipline is in the last two columns.

If you cannot state the design consequence, you probably have not converted the documentation into engineering understanding yet.


In [9]:
architecture_matrix = [
    {
        "statement": "Thinking/non-thinking control",
        "evidence": "Official model card",
        "capability": "Selectable reasoning behavior",
        "constraint": "Different latency/token behavior",
        "design_consequence": "Use a reasoning policy/router",
        "test": "Matched A/B task set",
    },
    {
        "statement": "Tool calling",
        "evidence": "Official model card + Qwen tool docs",
        "capability": "External action requests",
        "constraint": "Protocol/parser correctness matters",
        "design_consequence": "Model adapter + deterministic executor",
        "test": "Tool selection + argument accuracy",
    },
]
architecture_matrix


[{'statement': 'Thinking/non-thinking control',
  'evidence': 'Official model card',
  'capability': 'Selectable reasoning behavior',
  'constraint': 'Different latency/token behavior',
  'design_consequence': 'Use a reasoning policy/router',
  'test': 'Matched A/B task set'},
 {'statement': 'Tool calling',
  'evidence': 'Official model card + Qwen tool docs',
  'capability': 'External action requests',
  'constraint': 'Protocol/parser correctness matters',
  'design_consequence': 'Model adapter + deterministic executor',
  'test': 'Tool selection + argument accuracy'}]

# 10. Exercise: benchmark claims are evidence, not architecture

When a card says "state-of-the-art," "strong reasoning," or "leading agent performance," ask:

- Which benchmark?
- Which split?
- Which mode?
- What prompting?
- What tools?
- What evaluator?
- What inference settings?
- How close is the benchmark to my task?

A benchmark can establish evidence that a capability exists.

It cannot establish that:
- your tool interface is correct
- your retrieval is grounded
- your executor is safe
- your agent terminates
- your context strategy is sound
- your latency/cost target is met

### Agent evaluation is trajectory evaluation

Measure both:

**Final outcome**
- correctness
- completeness
- grounding

**Trajectory**
- tool choice
- argument correctness
- unnecessary calls
- recovery
- number of steps
- latency
- tokens
- cost
- verification behavior


In [10]:
trajectory_metrics = {
    "task_success": 0.0,
    "tool_selection_accuracy": 0.0,
    "tool_argument_accuracy": 0.0,
    "unnecessary_tool_rate": 0.0,
    "tool_error_recovery_rate": 0.0,
    "avg_steps": 0.0,
    "p95_latency_ms": 0.0,
    "avg_input_tokens": 0.0,
    "avg_output_tokens": 0.0,
    "grounding_score": 0.0,
    "verification_rate": 0.0,
}
trajectory_metrics


{'task_success': 0.0,
 'tool_selection_accuracy': 0.0,
 'tool_argument_accuracy': 0.0,
 'unnecessary_tool_rate': 0.0,
 'tool_error_recovery_rate': 0.0,
 'avg_steps': 0.0,
 'p95_latency_ms': 0.0,
 'avg_input_tokens': 0.0,
 'avg_output_tokens': 0.0,
 'grounding_score': 0.0,
 'verification_rate': 0.0}

# 11. Exercise: build the model adapter boundary

The strongest model-agnostic architecture keeps model-specific behavior at the edge.

```text
+---------------- MODEL ADAPTER ----------------+
| chat template                                 |
| reasoning switch                              |
| tool schema                                   |
| tool-result parser                            |
| output parser                                 |
| stop/special-token behavior                   |
+------------------------------------------------+
                      |
                      v
+------------------ AGENT CORE ------------------+
| routing | planning | execution | state | check |
+-------------------------------------------------+
                      |
                      v
+------------------- TOOL LAYER -----------------+
| search | retrieval | APIs | code | DB | files |
+-------------------------------------------------+
```

This means replacing Qwen3-8B should mostly affect the **adapter**, not the conceptual agent core.

That is the architecture pattern to aim for.


In [11]:
from typing import Protocol

class ModelAdapter(Protocol):
    def format_messages(self, messages: list[dict], **kwargs) -> object: ...
    def configure_reasoning(self, mode: str) -> dict: ...
    def format_tools(self, tools: list[dict]) -> object: ...
    def parse_model_output(self, raw_output: object) -> dict: ...
    def parse_tool_calls(self, raw_output: object) -> list[dict]: ...

def qwen3_reasoning_config(mode: str) -> dict:
    if mode == "thinking":
        return {"enable_thinking": True, "temperature": 0.6, "top_p": 0.95, "top_k": 20}
    if mode == "non-thinking":
        return {"enable_thinking": False, "temperature": 0.7, "top_p": 0.8, "top_k": 20}
    raise ValueError("mode must be 'thinking' or 'non-thinking'")

print(qwen3_reasoning_config("thinking"))
print(qwen3_reasoning_config("non-thinking"))


{'enable_thinking': True, 'temperature': 0.6, 'top_p': 0.95, 'top_k': 20}
{'enable_thinking': False, 'temperature': 0.7, 'top_p': 0.8, 'top_k': 20}


# 12. Optional: Qwen3 local inference skeleton

This cell is intentionally not required for the conceptual exercises.

Use it when you have the appropriate hardware and current dependencies installed.

The official card recommends recent `transformers` and shows `apply_chat_template` with `enable_thinking`.

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

messages = [{"role": "user", "content": "Solve this carefully."}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

inputs = tokenizer([text], return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=2048,
)

answer = tokenizer.decode(
    outputs[0][len(inputs.input_ids[0]):],
    skip_special_tokens=True
)

print(answer)
```

For production serving, use the model's documented serving/parser configuration rather than assuming a generic server will handle reasoning/tool output correctly.


# 13. Generalization challenge: pretend Qwen3 does not exist

Now imagine you have a completely unfamiliar model card.

Call it **NewModel-X**.

You learn:

- 14B MoE
- 64K context
- no explicit thinking mode
- tool calling supported through a vendor-specific JSON schema
- structured output is strong but only under a particular template
- vision input is supported
- 4-bit quantization is officially published
- local serving exists
- benchmark claims are strong for document understanding but weak for coding

### Your challenge

Design the agent **without using the phrase "Qwen" anywhere**.

What changes?

- reasoning policy
- tool adapter
- retrieval strategy
- context budget
- modality routing
- task router
- evaluation suite

This exercise is the proof that you learned the method rather than the model.


In [12]:
new_model_architecture = {
    "router": "route visual-document tasks to multimodal path; coding tasks to a stronger coding model or specialized path",
    "reasoning": "no model-provided reasoning switch; use task-aware prompting and external verification",
    "tools": "wrap vendor JSON schema behind a ModelAdapter",
    "context": "64K available, but reserve budget for tool results and output; use retrieval for large corpora",
    "memory": "keep durable memory and working state outside the raw prompt",
    "verification": "strong document-grounding checks; code tasks should be delegated or benchmarked separately",
    "deployment": "use official 4-bit artifact if memory pressure matters and benchmark quality after quantization",
}
new_model_architecture


{'router': 'route visual-document tasks to multimodal path; coding tasks to a stronger coding model or specialized path',
 'reasoning': 'no model-provided reasoning switch; use task-aware prompting and external verification',
 'tools': 'wrap vendor JSON schema behind a ModelAdapter',
 'context': '64K available, but reserve budget for tool results and output; use retrieval for large corpora',
 'memory': 'keep durable memory and working state outside the raw prompt',
 'verification': 'strong document-grounding checks; code tasks should be delegated or benchmarked separately',
 'deployment': 'use official 4-bit artifact if memory pressure matters and benchmark quality after quantization'}

# 14. Capstone: architect an excellent but simple agent

Choose a real task.

Good examples:
- research assistant
- document analyst
- support agent
- codebase investigator
- data analyst
- operations assistant
- knowledge-base agent

### Deliverable 1 - Model bio
One page.

### Deliverable 2 - Capability matrix
10-15 rows.

### Deliverable 3 - Agent hypothesis
Complete:

> "This model is best used as ______ because ______. I will use ______ for reasoning, ______ for tool interaction, ______ for memory/context, and ______ for verification. The main risk is ______, so I will evaluate ______."

### Deliverable 4 - Smallest loop
Draw the exact control loop.

### Deliverable 5 - Adapter boundary
Write down which pieces are model-specific.

### Deliverable 6 - Evaluation
Define at least:
- 10 representative tasks
- 5 trajectory metrics
- 3 failure modes
- 1 latency/cost budget

### Deliverable 7 - Failure-driven redesign
Run the tasks. Do not redesign from intuition.
Redesign from observed failures.


# 15. The universal "model-card engineer" checklist

Before shipping an agent around any model:

```text
[ ] Exact checkpoint identified
[ ] Base / instruct / chat role understood
[ ] Architecture understood
[ ] Parameter / active-parameter implications understood
[ ] Context limit understood
[ ] Practical context budget designed
[ ] Reasoning behavior understood
[ ] Reasoning control understood (if available)
[ ] Tool calling protocol understood
[ ] Tool result protocol understood
[ ] Chat template understood
[ ] Structured output behavior understood
[ ] Serving/parser requirements understood
[ ] Sampling recommendations understood
[ ] Quantization implications understood
[ ] Limitations and failure modes identified
[ ] Benchmark evidence interrogated
[ ] Agent loop reduced to the smallest useful loop
[ ] Model-specific logic isolated in an adapter
[ ] Trajectory-level evaluation designed
[ ] Cost / latency measured
```

The final question:

> **Which facts from the model card actually change my architecture?**

That question is the habit to keep.


# 16. Reflection

Write this in your own words after completing the workbook:

1. What is the difference between a model capability and an agent capability?
2. Why is the chat template part of the model interface?
3. Why should reasoning be a policy decision rather than a default?
4. Why is context length not the same as memory?
5. Why do tool calling benchmarks not prove an agent is reliable?
6. Which pieces of your architecture should remain model-agnostic?
7. What would make you replace a model without rewriting your whole agent?

### The target outcome

You should be able to open an unfamiliar model card and, within one focused session, move from:

**"What is this model?"**

to:

**"Here is the simplest agent architecture that exploits its strengths, avoids its weak points, isolates its quirks, and proves its value with measurable tests."**
